# 03. 모델링 v2 — 피처 정리 + Optuna 튜닝
**베이스라인 결과 (v1)**
- LightGBM OOF AUC: 0.7328
- XGBoost OOF AUC: 0.7356

**v2 목표**
1. 임포턴스 0 또는 매우 낮은 피처 제거
2. Optuna로 XGBoost 하이퍼파라미터 튜닝
3. 튜닝된 LightGBM도 추가 비교

## 0. 환경 세팅

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 한글 폰트
fm.fontManager.addfont('/System/Library/Fonts/Supplemental/AppleGothic.ttf')
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

os.chdir('/Users/admin/Desktop/hackerton/Fertility-Success-Predictor')

RANDOM_STATE = 42
N_FOLDS = 5

In [2]:
X      = pd.read_csv('data/X_train.csv')
y      = pd.read_csv('data/y_train.csv').squeeze()
X_test = pd.read_csv('data/X_test.csv')
test_ids = pd.read_csv('data/test_ids.csv').squeeze()

SCALE_POS_WEIGHT = (y==0).sum() / (y==1).sum()
print(f'X: {X.shape}  |  scale_pos_weight: {SCALE_POS_WEIGHT:.2f}')

X: (256351, 101)  |  scale_pos_weight: 2.87


## 1. 피처 정리

In [3]:
# v1 임포턴스 기반 제거 대상
# - 배란유도_기록됨: 임포턴스 0
# - 현재시술_목적: 임포턴스 20 (최하위권)
# - 저반응_여부, 고반응_여부: 22~32 (성숙_난자율에 이미 포함된 정보)
# - 불임 원인 세부 (자궁경부 문제, 정자 면역학적, 정자 농도 등): 극히 낮음

DROP_FEATURES = [
    '배란유도_기록됨',
    '현재시술_목적',
    '저반응_여부',
    '고반응_여부',
    '불임 원인 - 자궁경부 문제',
    '불임 원인 - 정자 면역학적 요인',
    '불임 원인 - 정자 농도',
    '불임 원인 - 정자 운동성',
    '불임 원인 - 정자 형태',
]
# 실제 존재하는 컬럼만 제거
DROP_FEATURES = [c for c in DROP_FEATURES if c in X.columns]

X      = X.drop(columns=DROP_FEATURES)
X_test = X_test.drop(columns=DROP_FEATURES)

print(f'피처 제거 후 → X: {X.shape}  |  X_test: {X_test.shape}')
print(f'제거된 피처: {DROP_FEATURES}')

피처 제거 후 → X: (256351, 92)  |  X_test: (90067, 92)
제거된 피처: ['배란유도_기록됨', '현재시술_목적', '저반응_여부', '고반응_여부', '불임 원인 - 자궁경부 문제', '불임 원인 - 정자 면역학적 요인', '불임 원인 - 정자 농도', '불임 원인 - 정자 운동성', '불임 원인 - 정자 형태']


## 2. 교차검증 함수

In [4]:
def target_encode(X_tr, y_tr, X_val, X_te, cols, smooth=20):
    """
    CV 내부에서만 fit → leakage 없음
    smooth: 전체 평균으로 스무딩 (소규모 카테고리 과적합 방지)
    """
    X_tr  = X_tr.copy()
    X_val = X_val.copy()
    X_te  = X_te.copy()
    global_mean = y_tr.mean()

    for col in cols:
        # train fold 통계
        stats = y_tr.groupby(X_tr[col]).agg(['mean', 'count'])
        # 스무딩: (count * mean + smooth * global_mean) / (count + smooth)
        smoothed = (stats['count'] * stats['mean'] + smooth * global_mean) / (stats['count'] + smooth)

        X_tr[f'{col}_te']  = X_tr[col].map(smoothed).fillna(global_mean)
        X_val[f'{col}_te'] = X_val[col].map(smoothed).fillna(global_mean)
        X_te[f'{col}_te']  = X_te[col].map(smoothed).fillna(global_mean)

    return X_tr, X_val, X_te


# Target Encoding 적용할 컬럼
TE_COLS = ['시술 시기 코드', '특정 시술 유형', '배아 생성 주요 이유']


def cross_validate(model_fn, X, y, n_folds=N_FOLDS, verbose=True, use_te=True):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    oof_preds  = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx].copy(), X.iloc[val_idx].copy()
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        X_te = X_test.copy()

        # CV 내부에서 Target Encoding (leakage 없음)
        if use_te:
            X_tr, X_val, X_te = target_encode(X_tr, y_tr, X_val, X_te, TE_COLS)

        model = model_fn(fold)
        model.fit(X_tr, y_tr)

        val_pred  = model.predict_proba(X_val)[:, 1]
        test_pred = model.predict_proba(X_te)[:, 1]

        oof_preds[val_idx] = val_pred
        test_preds += test_pred / n_folds

        score = roc_auc_score(y_val, val_pred)
        scores.append(score)
        if verbose:
            print(f'  Fold {fold+1} ROC-AUC: {score:.4f}')

    oof_score = roc_auc_score(y, oof_preds)
    if verbose:
        print(f'  CV 평균: {np.mean(scores):.4f} ± {np.std(scores):.4f}')
        print(f'  OOF ROC-AUC: {oof_score:.4f}')

    return oof_preds, test_preds, scores, oof_score


## 3. Optuna 튜닝 — XGBoost

In [5]:
# Optuna: 50 trial, 3-fold (속도용) → 최적 파라미터 탐색
# 튜닝 완료 후 5-fold로 최종 평가

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 5.0, log=True),
        'scale_pos_weight': SCALE_POS_WEIGHT,
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'verbosity': 0,
        'eval_metric': 'auc',
    }

    # 튜닝 시 3-fold 사용 (속도)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr)
        pred = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, pred))
    return np.mean(scores)

print('XGBoost Optuna 튜닝 시작 (50 trials × 3-fold)...')
xgb_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
xgb_study.optimize(xgb_objective, n_trials=50)

print(f'\n최적 AUC: {xgb_study.best_value:.4f}')
print(f'최적 파라미터:')
for k, v in xgb_study.best_params.items():
    print(f'  {k}: {v}')

XGBoost Optuna 튜닝 시작 (50 trials × 3-fold)...

최적 AUC: 0.7397
최적 파라미터:
  n_estimators: 971
  learning_rate: 0.017207463958574424
  max_depth: 4
  min_child_weight: 4
  subsample: 0.6448496799947323
  colsample_bytree: 0.8976569684739044
  reg_alpha: 0.5206947277258029
  reg_lambda: 3.5722571172202193


## 4. Optuna 튜닝 — LightGBM

In [6]:
def lgbm_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 5.0, log=True),
        'scale_pos_weight': SCALE_POS_WEIGHT,
        'random_state': RANDOM_STATE,
        'n_jobs': -1,
        'verbose': -1,
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr)
        pred = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, pred))
    return np.mean(scores)

print('LightGBM Optuna 튜닝 시작 (50 trials × 3-fold)...')
lgbm_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
lgbm_study.optimize(lgbm_objective, n_trials=50)

print(f'\n최적 AUC: {lgbm_study.best_value:.4f}')
print(f'최적 파라미터:')
for k, v in lgbm_study.best_params.items():
    print(f'  {k}: {v}')

LightGBM Optuna 튜닝 시작 (50 trials × 3-fold)...

최적 AUC: 0.7393
최적 파라미터:
  n_estimators: 799
  learning_rate: 0.011548439019600258
  num_leaves: 127
  max_depth: 6
  min_child_samples: 47
  subsample: 0.6646002324413683
  colsample_bytree: 0.6152042892278028
  reg_alpha: 0.04469576772939621
  reg_lambda: 1.4458814297668934


## 5. 튜닝된 모델로 5-fold 최종 평가

In [7]:
# XGBoost 튜닝 버전
best_xgb_params = {**xgb_study.best_params,
                   'scale_pos_weight': SCALE_POS_WEIGHT,
                   'random_state': RANDOM_STATE,
                   'n_jobs': -1, 'verbosity': 0, 'eval_metric': 'auc'}

print('=== XGBoost (튜닝) ===')
xgb_oof, xgb_test, xgb_scores, xgb_oof_score = cross_validate(
    lambda fold: XGBClassifier(**best_xgb_params), X, y
)

=== XGBoost (튜닝) ===
  Fold 1 ROC-AUC: 0.7383
  Fold 2 ROC-AUC: 0.7429
  Fold 3 ROC-AUC: 0.7400
  Fold 4 ROC-AUC: 0.7384
  Fold 5 ROC-AUC: 0.7413
  CV 평균: 0.7402 ± 0.0018
  OOF ROC-AUC: 0.7401


In [8]:
# LightGBM 튜닝 버전
best_lgbm_params = {**lgbm_study.best_params,
                    'scale_pos_weight': SCALE_POS_WEIGHT,
                    'random_state': RANDOM_STATE,
                    'n_jobs': -1, 'verbose': -1}

print('=== LightGBM (튜닝) ===')
lgbm_oof, lgbm_test, lgbm_scores, lgbm_oof_score = cross_validate(
    lambda fold: LGBMClassifier(**best_lgbm_params), X, y
)

=== LightGBM (튜닝) ===
  Fold 1 ROC-AUC: 0.7379
  Fold 2 ROC-AUC: 0.7426
  Fold 3 ROC-AUC: 0.7397
  Fold 4 ROC-AUC: 0.7384
  Fold 5 ROC-AUC: 0.7408
  CV 평균: 0.7399 ± 0.0017
  OOF ROC-AUC: 0.7399


## 5. Optuna 튜닝 — CatBoost

In [9]:
from catboost import CatBoostClassifier

def catboost_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 300, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 5.0, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        'scale_pos_weight': SCALE_POS_WEIGHT,
        'random_seed': RANDOM_STATE,
        'verbose': 0,
    }
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr_idx, val_idx in skf.split(X, y):
        model = CatBoostClassifier(**params)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        pred = model.predict_proba(X.iloc[val_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[val_idx], pred))
    return np.mean(scores)

print('CatBoost Optuna 튜닝 시작 (30 trials × 3-fold)...')
cat_study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
cat_study.optimize(catboost_objective, n_trials=30)

print(f'최적 AUC: {cat_study.best_value:.4f}')
print(f'최적 파라미터:')
for k, v in cat_study.best_params.items():
    print(f'  {k}: {v}')

CatBoost Optuna 튜닝 시작 (30 trials × 3-fold)...
최적 AUC: 0.7396
최적 파라미터:
  iterations: 818
  learning_rate: 0.02355526186331552
  depth: 6
  l2_leaf_reg: 2.8620802834265953
  subsample: 0.797034766206513
  colsample_bylevel: 0.9178581456448356


In [10]:
# CatBoost 5-fold 최종 평가
best_cat_params = {**cat_study.best_params,
                   'scale_pos_weight': SCALE_POS_WEIGHT,
                   'random_seed': RANDOM_STATE,
                   'verbose': 0}

print('=== CatBoost (튜닝) ===')
cat_oof, cat_test, cat_scores, cat_oof_score = cross_validate(
    lambda fold: CatBoostClassifier(**best_cat_params), X, y
)

=== CatBoost (튜닝) ===
  Fold 1 ROC-AUC: 0.7382
  Fold 2 ROC-AUC: 0.7428
  Fold 3 ROC-AUC: 0.7400
  Fold 4 ROC-AUC: 0.7386
  Fold 5 ROC-AUC: 0.7409
  CV 평균: 0.7401 ± 0.0017
  OOF ROC-AUC: 0.7401


In [11]:
# 결과 비교
print('=' * 55)
print(f'{"모델":<20} {"OOF AUC":>10} {"CV 평균":>10}')
print('-' * 55)
print(f'{"LightGBM (v1)":<20} {0.7328:>10.4f} {0.7328:>10.4f}')
print(f'{"XGBoost (v1)":<20} {0.7356:>10.4f} {0.7356:>10.4f}')
print('-' * 55)
print(f'{"LightGBM (튜닝)":<20} {lgbm_oof_score:>10.4f} {np.mean(lgbm_scores):>10.4f}')
print(f'{"XGBoost (튜닝)":<20} {xgb_oof_score:>10.4f} {np.mean(xgb_scores):>10.4f}')
print(f'{"CatBoost (튜닝)":<20} {cat_oof_score:>10.4f} {np.mean(cat_scores):>10.4f}')
print('=' * 55)

모델                      OOF AUC      CV 평균
-------------------------------------------------------
LightGBM (v1)            0.7328     0.7328
XGBoost (v1)             0.7356     0.7356
-------------------------------------------------------
LightGBM (튜닝)            0.7399     0.7399
XGBoost (튜닝)             0.7401     0.7402
CatBoost (튜닝)            0.7401     0.7401


## 6. 제출 파일 생성

In [12]:
# 3모델 앙상블 가중치 탐색
print('=== 3모델 앙상블 가중치 탐색 ===')
best_auc, best_w = 0, (0.33, 0.33, 0.34)

for w1 in np.arange(0.1, 0.9, 0.1):
    for w2 in np.arange(0.1, 0.9, 0.1):
        w3 = round(1 - w1 - w2, 1)
        if w3 <= 0 or w3 >= 1:
            continue
        ensemble = w1 * lgbm_oof + w2 * xgb_oof + w3 * cat_oof
        score = roc_auc_score(y, ensemble)
        if score > best_auc:
            best_auc = score
            best_w = (w1, w2, w3)

w1, w2, w3 = best_w
print(f'최적 가중치: LGBM {w1:.1f} / XGB {w2:.1f} / CAT {w3:.1f}')
print(f'앙상블 OOF AUC: {best_auc:.4f}')
print(f'단일 모델 비교:')
print(f'  LightGBM: {lgbm_oof_score:.4f}')
print(f'  XGBoost:  {xgb_oof_score:.4f}')
print(f'  CatBoost: {cat_oof_score:.4f}')
print(f'  앙상블:   {best_auc:.4f}')

# 최종 제출
final_preds = w1 * lgbm_test + w2 * xgb_test + w3 * cat_test
submission = pd.DataFrame({'ID': test_ids, 'probability': final_preds})
os.makedirs('submission', exist_ok=True)
filename = f'submission/submission_ensemble_v3_200trials.csv'
submission.to_csv(filename, index=False)
print(f'저장 완료 → {filename}')
submission.head()

=== 3모델 앙상블 가중치 탐색 ===
최적 가중치: LGBM 0.2 / XGB 0.4 / CAT 0.4
앙상블 OOF AUC: 0.7403
단일 모델 비교:
  LightGBM: 0.7399
  XGBoost:  0.7401
  CatBoost: 0.7401
  앙상블:   0.7403
저장 완료 → submission/submission_ensemble_v3_200trials.csv


,ID,probability
0,TEST_00000,0.004989
1,TEST_00001,0.003580
2,TEST_00002,0.347285
3,TEST_00003,0.274060
4,TEST_00004,0.740184
